# Phase 2: Data Cleaning & Standardization
## Avoidable Emergency Department Utilization Navigator

This notebook cleans and standardizes raw CMS Synthetic Medicare Enrollment and FFS Claims datasets (`beneficiary_2022.csv`, `inpatient.csv`, and `outpatient.csv`).

### ⚠️ Healthcare Data Interpretation Disclaimer:
> **CRITICAL DIRECTIVE:** Missing financial fields (such as deductibles, co-pays, or patient liability metrics) were normalized to `0.0` strictly for technical processing and calculation stability during MVP analytics. **These normalized values should NOT be interpreted as confirmed zero patient responsibility or verified zero cost sharing.** In real-world CMS claims processing, null financial fields may indicate non-adjudicated lines or missing benefit coordination records.

### Key Directives & Principles:
1. **Preserve Healthcare Meaning**: Original CMS coding, revenue center codes (`0450` for ED), DRG codes, and ICD-10 diagnosis codes are strictly preserved.
2. **No Arbitrary Filters**: Claims are never dropped without explicit justification (e.g., unmatched beneficiary ID).
3. **No Artificial Classifications**: Outpatient visits are not re-labeled unless supported by official revenue center / HCPCS coding.
4. **Output Integrity**: Cleaned files saved to `processed_data/` (`beneficiary_clean.csv`, `inpatient_clean.csv`, `outpatient_clean.csv`).

### Step 1: Load Raw Datasets

In [ ]:
import pandas as pd
import numpy as np
import os

raw_dir = '../datasets'
processed_dir = '../processed_data'
os.makedirs(processed_dir, exist_ok=True)

# Load raw pipe-delimited files
df_bene = pd.read_csv(os.path.join(raw_dir, 'beneficiary_2022.csv'), sep='|', low_memory=False)
df_inp = pd.read_csv(os.path.join(raw_dir, 'inpatient.csv'), sep='|', low_memory=False)
df_outp = pd.read_csv(os.path.join(raw_dir, 'outpatient.csv'), sep='|', low_memory=False)

print(f"Raw Beneficiary Records: {len(df_bene):,} rows, {df_bene.shape[1]} columns")
print(f"Raw Inpatient Claims: {len(df_inp):,} rows, {df_inp.shape[1]} columns")
print(f"Raw Outpatient Claims: {len(df_outp):,} rows, {df_outp.shape[1]} columns")

### Step 2: Clean Beneficiary Data (`beneficiary_clean.csv`)

**Transformations applied:**
- Strip leading/trailing whitespace from `BENE_ID`.
- Parse `BENE_BIRTH_DT` and `BENE_DEATH_DT` to standard `YYYY-MM-DD` ISO format.
- Deduplicate unique beneficiary keys.

In [ ]:
df_bene['BENE_ID'] = df_bene['BENE_ID'].astype(str).str.strip()
df_bene['BENE_BIRTH_DT'] = pd.to_datetime(df_bene['BENE_BIRTH_DT'], errors='coerce').dt.strftime('%Y-%m-%d')

if 'BENE_DEATH_DT' in df_bene.columns:
    df_bene['BENE_DEATH_DT'] = pd.to_datetime(df_bene['BENE_DEATH_DT'], errors='coerce').dt.strftime('%Y-%m-%d')

init_bene_count = len(df_bene)
df_bene = df_bene.drop_duplicates(subset=['BENE_ID']).reset_index(drop=True)
print(f"Beneficiary deduplication complete: {init_bene_count - len(df_bene)} duplicates removed.")

### Step 3: Clean Inpatient Claims Data (`inpatient_clean.csv`)

**Transformations applied:**
- Strip whitespace from key fields (`BENE_ID`, `CLM_ID`, `REV_CNTR`, `ADMTG_DGNS_CD`, `PRNCPAL_DGNS_CD`).
- Parse date fields (`CLM_FROM_DT`, `CLM_THRU_DT`, `CLM_ADMSN_DT`) to ISO format (`YYYY-MM-DD`).
- Standardize numeric cost fields (`CLM_PMT_AMT`, `CLM_TOT_CHRG_AMT`, `NCH_BENE_IP_DDCTBL_AMT`, `NCH_BENE_PTA_COINSRNC_LBLTY_AM`), normalizing NaNs to `0.0` for technical processing stability.
- Validate `BENE_ID` against clean beneficiary master table.

In [ ]:
df_inp['BENE_ID'] = df_inp['BENE_ID'].astype(str).str.strip()
df_inp['CLM_ID'] = df_inp['CLM_ID'].astype(str).str.strip()

for dt_col in ['CLM_FROM_DT', 'CLM_THRU_DT', 'CLM_ADMSN_DT']:
    if dt_col in df_inp.columns:
        df_inp[dt_col] = pd.to_datetime(df_inp[dt_col], errors='coerce').dt.strftime('%Y-%m-%d')

numeric_cols_inp = ['CLM_PMT_AMT', 'CLM_TOT_CHRG_AMT', 'NCH_BENE_IP_DDCTBL_AMT', 'NCH_BENE_PTA_COINSRNC_LBLTY_AM']
for col in numeric_cols_inp:
    if col in df_inp.columns:
        df_inp[col] = pd.to_numeric(df_inp[col], errors='coerce').fillna(0.0)

valid_bene_ids = set(df_bene['BENE_ID'])
init_inp_count = len(df_inp)
df_inp = df_inp[df_inp['BENE_ID'].isin(valid_bene_ids)].reset_index(drop=True)
print(f"Inpatient claim validation complete: {init_inp_count - len(df_inp)} orphan claims dropped.")

### Step 4: Clean Outpatient Claims Data (`outpatient_clean.csv`)

**Transformations applied:**
- Strip whitespace from key fields (`BENE_ID`, `CLM_ID`, `REV_CNTR`, `PRNCPAL_DGNS_CD`, `HCPCS_CD`).
- Parse service encounter dates (`CLM_FROM_DT`, `CLM_THRU_DT`) to ISO format (`YYYY-MM-DD`).
- Standardize numeric cost fields (`CLM_PMT_AMT`, `CLM_TOT_CHRG_AMT`, `NCH_BENE_PTB_DDCTBL_AMT`), normalizing NaNs to `0.0` for processing stability.
- Validate `BENE_ID` against clean beneficiary master table.

In [ ]:
df_outp['BENE_ID'] = df_outp['BENE_ID'].astype(str).str.strip()
df_outp['CLM_ID'] = df_outp['CLM_ID'].astype(str).str.strip()

for dt_col in ['CLM_FROM_DT', 'CLM_THRU_DT']:
    if dt_col in df_outp.columns:
        df_outp[dt_col] = pd.to_datetime(df_outp[dt_col], errors='coerce').dt.strftime('%Y-%m-%d')

numeric_cols_outp = ['CLM_PMT_AMT', 'CLM_TOT_CHRG_AMT', 'NCH_BENE_PTB_DDCTBL_AMT', 'NCH_BENE_PTB_COINSRNC_AMT']
for col in numeric_cols_outp:
    if col in df_outp.columns:
        df_outp[col] = pd.to_numeric(df_outp[col], errors='coerce').fillna(0.0)

init_outp_count = len(df_outp)
df_outp = df_outp[df_outp['BENE_ID'].isin(valid_bene_ids)].reset_index(drop=True)
print(f"Outpatient claim validation complete: {init_outp_count - len(df_outp)} orphan claims dropped.")

### Step 5: Save Cleaned Datasets & Verification

In [ ]:
bene_clean_path = os.path.join(processed_dir, 'beneficiary_clean.csv')
inp_clean_path = os.path.join(processed_dir, 'inpatient_clean.csv')
outp_clean_path = os.path.join(processed_dir, 'outpatient_clean.csv')

df_bene.to_csv(bene_clean_path, index=False)
df_inp.to_csv(inp_clean_path, index=False)
df_outp.to_csv(outp_clean_path, index=False)

print(f"Cleaned Beneficiary File: {bene_clean_path} ({os.path.getsize(bene_clean_path)/(1024*1024):.2f} MB, {len(df_bene):,} rows)")
print(f"Cleaned Inpatient File: {inp_clean_path} ({os.path.getsize(inp_clean_path)/(1024*1024):.2f} MB, {len(df_inp):,} rows)")
print(f"Cleaned Outpatient File: {outp_clean_path} ({os.path.getsize(outp_clean_path)/(1024*1024):.2f} MB, {len(df_outp):,} rows)")